# ESI HV module activation probe

This notebook isolates the three independent HV conditions exposed by `COM-ESI-CTRL.dll`: target voltage, controller-wide `SetEnable`, and module-level `SetModuleActivationState`. It tests whether the module toggle depends on the controller-wide gate.

## Safety

- Run on the Windows PC connected to the ESI controller.
- First run the read-only `ESI-Controller.exe` preflight from the vendor package's top-level `Software` folder.
- Close ESIBD Explorer and all vendor utilities before this probe; the DLL is single-instance.
- Keep the hardware interlock available and use an external meter.
- Every run first writes `0 V` to both HV modules, tests the module toggle with the global gate OFF, resets to standby, then repeats with the global gate ON.
- The probe does not restore previous output states. Its finalizer writes `0 V`, requests module standby, disables the global gate, and closes the COM port.
- Configuration writes and nonzero output have separate arm switches, both disabled by default. When explicitly armed, `ARM_TEMP_CONFIG=True` changes only the selected module's `HVPSxMaxVoltStep` from the verified-safe initial configuration; the guarded path stages `100 V` while the module is OFF, allows up to 10 seconds to reach a `95 V` PWM trigger, then records a 20-second hold under the operator-approved absolute `150 V` cutoff. The trigger starts observation only; accuracy is recorded without an automatic nominal-tolerance verdict.
- The temporary configuration is accepted only when the initial 53-byte configuration is demonstrably safe. The finalizer first forces all targets and gates OFF, then restores and verifies those original safe bytes.
- The vendor DLL calls are not cancellable. Restart the Jupyter kernel if a call blocks.


In [ ]:
# Adjust these values only while ESIBD Explorer is closed.
COM_PORT = 16
BAUD_RATE = 230400
MODULE_ADDRESS = 1  # HV1=1, HV2=2
ARM_TEMP_CONFIG = False
ARM_NONZERO_TEST = False
TEST_VOLTAGE = 100.0
MAX_VOLTAGE_STEP = 10.008
MAX_VOLTAGE_STEP_RAW = 10008
SETTLE_SECONDS = 2.0
ZERO_ADC_ABS_LIMIT_V = 1.0
NONZERO_RISE_TIMEOUT_SECONDS = 10.0
NONZERO_HOLD_SECONDS = 20.0
NONZERO_HOLD_START_V = 95.0
NONZERO_POLL_SECONDS = 0.05
NONZERO_ABS_LIMIT_V = 150.0
NONZERO_ADC_GRACE_SECONDS = 1.5
DISCHARGE_LIMIT_V = 1.0
DISCHARGE_TIMEOUT_SECONDS = 60.0
DISCHARGE_POLL_SECONDS = 0.25

from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / 'esi',
    Path.home() / 'ESIBD Explorer' / 'plugins' / 'esi',
]
PLUGIN_DIR = next(
    (path for path in candidates if (path / 'esi_plugin.py').is_file()),
    None,
)
if PLUGIN_DIR is None:
    raise FileNotFoundError(
        'Could not locate the installed esi plugin folder. Start Jupyter '
        'from the plugins or esi folder.'
    )

DRIVER_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'esi_base.py'
DLL_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'vendor' / 'x64' / 'COM-ESI-CTRL.dll'
ERROR_CODES_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'error_codes.json'
print(f'Plugin: {PLUGIN_DIR}')
print(f'COM{COM_PORT} @ {BAUD_RATE} baud; module={MODULE_ADDRESS}')
print(f'Temporary configuration armed: {ARM_TEMP_CONFIG}')
print(f'Nonzero test armed: {ARM_NONZERO_TEST}')


In [ ]:
import ctypes
import hashlib
import importlib.util
import json
import math
import struct
import time
from datetime import datetime

spec = importlib.util.spec_from_file_location('_esi_activation_probe_base', DRIVER_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load {DRIVER_FILE}')
driver_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver_module)
ESIBase = driver_module.ESIBase
# Layout used by the manufacturer utility: <target mV><max-step mV><3 flags><pad>.
HV_CONFIG_BASE_OFFSET = 17
HV_CONFIG_STRIDE = 12
HV_MAX_STEP_RELATIVE_OFFSET = 4
EXPECTED_DLL_SHA256 = '991637c4dab5ed6d2801543696b462c038c9d7504caef0f86d9a2ba208232456'
DLL_SHA256 = hashlib.sha256(DLL_FILE.read_bytes()).hexdigest()
if DLL_SHA256 != EXPECTED_DLL_SHA256:
    raise RuntimeError(
        f'Wrong COM-ESI-CTRL.dll: expected {EXPECTED_DLL_SHA256}, got {DLL_SHA256}'
    )

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def record_call(device, action, method, *args, required=False):
    started = time.monotonic()
    try:
        result = method(*args)
        if isinstance(result, tuple):
            status, values = int(result[0]), list(result[1:])
        else:
            status, values = int(result), []
        record = {
            'action': action,
            'status': status,
            'status_text': device.format_status(status),
            'duration_s': round(time.monotonic() - started, 6),
            'values': json_safe(values),
        }
    except Exception as exc:
        record = {
            'action': action,
            'error': f'{type(exc).__name__}: {exc}',
            'duration_s': round(time.monotonic() - started, 6),
        }
        if required:
            raise
        return record
    if required and status != device.NO_ERR:
        raise RuntimeError(f'{action} failed: {device.format_status(status)}')
    return record

def read_module_led(device, address):
    method = getattr(device, 'get_module_led_data', None)
    if callable(method):
        return method(address)
    red = ctypes.c_bool()
    green = ctypes.c_bool()
    blue = ctypes.c_bool()
    function = device.esi_dll.COM_ESI_CTRL_GetModuleLEDData
    function.argtypes = [
        ctypes.c_uint,
        ctypes.POINTER(ctypes.c_bool),
        ctypes.POINTER(ctypes.c_bool),
        ctypes.POINTER(ctypes.c_bool),
    ]
    function.restype = ctypes.c_int
    status = function(
        ctypes.c_uint(address),
        ctypes.byref(red),
        ctypes.byref(green),
        ctypes.byref(blue),
    )
    return status, red.value, green.value, blue.value

def read_hv_state(device, address):
    state = {
        'global_enable': record_call(device, 'get_enable', device.get_enable),
        'complete_state': record_call(
            device, 'get_complete_state', device.get_complete_state
        ),
        'main_state': record_call(device, 'get_main_state', device.get_main_state),
        'device_state': record_call(device, 'get_device_state', device.get_device_state),
        'voltage_state': record_call(device, 'get_voltage_state', device.get_voltage_state),
        'temperature_state': record_call(
            device, 'get_temperature_state', device.get_temperature_state
        ),
        'fan_state': record_call(device, 'get_fan_state', device.get_fan_state),
        'interlock_state': record_call(device, 'get_interlock_state', device.get_interlock_state),
        'interlock_enable': record_call(device, 'get_interlock_enable', device.get_interlock_enable),
        'inputs': record_call(device, 'get_inputs', device.get_inputs),
        'power_monitors': record_call(
            device, 'get_power_monitors', device.get_power_monitors
        ),
        'module_state': record_call(device, 'get_module_state', device.get_module_state, address),
        'module_activation_direct': record_call(
            device, 'get_module_activation_state',
            device.get_module_activation_state, address
        ),
        'module_led': record_call(
            device, 'get_module_led_data', read_module_led, device, address
        ),
        'measurement_ranges': record_call(
            device, 'get_hv_supply_meas_ranges',
            device.get_hv_supply_meas_ranges, address
        ),
        'pwm': record_call(device, 'get_hv_supply_params_pwm', device.get_hv_supply_params_pwm, address),
        'target': record_call(
            device, 'get_hv_supply_target_output_voltage',
            device.get_hv_supply_target_output_voltage, address
        ),
        'measured_voltage': record_call(
            device, 'get_hv_supply_output_voltage',
            device.get_hv_supply_output_voltage, address
        ),
        'measured_current': record_call(
            device, 'get_hv_supply_output_current',
            device.get_hv_supply_output_current, address
        ),
    }
    def values_if_ok(name):
        record = state[name]
        return (
            record.get('values', [])
            if record.get('status') == device.NO_ERR else []
        )

    pwm_values = values_if_ok('pwm')
    state['pwm_activation'] = bool(pwm_values[6]) if len(pwm_values) >= 7 else None
    activation_values = values_if_ok('module_activation_direct')
    state['direct_activation'] = (
        bool(activation_values[0]) if activation_values else None
    )
    module_state_values = values_if_ok('module_state')
    module_state = int(module_state_values[0]) if module_state_values else None
    state['module_state_raw'] = module_state
    module_state_text = (
        f'0x{module_state:04X}' if module_state is not None else 'n/a'
    )
    state['module_state_active'] = (
        bool(module_state & device.MS_ACTIVE) if module_state is not None else None
    )
    state['control_active'] = (
        bool(module_state & device.MS_CTRL_ACT) if module_state is not None else None
    )
    state['module_gate_active'] = (
        bool(module_state & device.MS_MOD_ACT) if module_state is not None else None
    )
    state['device_gate_active'] = (
        bool(module_state & device.MS_DEV_ACT) if module_state is not None else None
    )
    state['pwm_period_s'] = float(pwm_values[0]) if len(pwm_values) >= 1 else None
    state['pwm_width_s'] = float(pwm_values[1]) if len(pwm_values) >= 2 else None
    state['pwm_voltage_set_v'] = float(pwm_values[4]) if len(pwm_values) >= 5 else None
    state['pwm_voltage_measured_v'] = float(pwm_values[5]) if len(pwm_values) >= 6 else None
    target_values = values_if_ok('target')
    state['target_v'] = float(target_values[0]) if target_values else None
    enable_values = values_if_ok('global_enable')
    state['global_enabled'] = bool(enable_values[0]) if enable_values else None
    led_values = values_if_ok('module_led')
    state['module_led_rgb'] = led_values[:3] if len(led_values) >= 3 else None
    print(
        f"global={state['global_enabled']}, "
        f"module_direct_active={state['direct_activation']}, "
        f"module_pwm_active={state['pwm_activation']}, "
        f"module_state={module_state_text}, "
        f"control_active={state['control_active']}, "
        f"target={state['target_v']} V, "
        f"pwm_set={state['pwm_voltage_set_v']} V, "
        f"pwm_measured={state['pwm_voltage_measured_v']} V, "
        f"LED={state['module_led_rgb']}, "
        f"interlock={state['interlock_state'].get('values')}"
    )
    return state

def select_and_read_adc(
    device, address, negative, current_high,
    safety_limit_v=None, abort_callback=None,
):
    polarity = 'negative' if negative else 'positive'
    result = {
        'selection_request': record_call(
            device, f'select_{polarity}_adc',
            device.set_hv_supply_meas_ranges,
            address, bool(negative), bool(current_high), required=True,
        )
    }
    if safety_limit_v is None:
        time.sleep(SETTLE_SECONDS)
    else:
        settle_started = time.monotonic()
        settle_guard_samples = []
        while True:
            pwm = record_call(
                device, f'guard_{polarity}_adc_settle_pwm',
                device.get_hv_supply_params_pwm, address, required=True,
            )
            pwm_values = pwm.get('values', [])
            elapsed = time.monotonic() - settle_started
            sample = {
                'elapsed_s': round(elapsed, 6),
                'pwm': pwm,
            }
            settle_guard_samples.append(sample)
            reason = None
            if len(pwm_values) < 7:
                reason = f'incomplete PWM during {polarity} ADC settling'
            elif not bool(pwm_values[6]):
                reason = f'PWM activation was lost during {polarity} ADC settling'
            else:
                pwm_measured_v = float(pwm_values[5])
                sample['pwm_measured_v'] = pwm_measured_v
                if (
                    not math.isfinite(pwm_measured_v)
                    or abs(pwm_measured_v) > float(safety_limit_v)
                ):
                    reason = (
                        f'{polarity} ADC settling PWM magnitude '
                        f'{abs(pwm_measured_v):g} V exceeds '
                        f'{float(safety_limit_v):g} V'
                    )
            if reason is not None:
                if abort_callback is not None:
                    abort_callback(reason)
                raise RuntimeError(reason)
            if elapsed >= SETTLE_SECONDS:
                break
            time.sleep(min(NONZERO_POLL_SECONDS, SETTLE_SECONDS - elapsed))
        result['settle_guard_samples'] = settle_guard_samples
    result['selection_readback'] = record_call(
        device, f'verify_{polarity}_adc',
        device.get_hv_supply_meas_ranges, address, required=True,
    )
    observed = result['selection_readback'].get('values', [])
    requested = [bool(negative), bool(current_high)]
    if observed[:2] != requested:
        raise RuntimeError(
            f'{polarity} ADC selection failed: requested {requested}, got {observed[:2]}'
        )
    result['state'] = read_hv_state(device, address)
    return result

def decode_current_config(raw_config):
    raw = bytes(raw_config)
    if len(raw) != ESIBase.CONFIG_DATA_SIZE:
        raise RuntimeError(
            f'Expected {ESIBase.CONFIG_DATA_SIZE} configuration bytes, got {len(raw)}'
        )

    def milli_units(offset):
        return float(struct.unpack_from('<i', raw, offset)[0]) / 1000.0

    def heater_limit(offset):
        return float(struct.unpack_from('<H', raw, offset)[0]) * 1024.0 / 1e6

    def heater_power(offset):
        value = struct.unpack_from('<I', raw, offset)[0]
        return float(value) * 1024.0**3 / 1e12

    decoded = {
        'device_enabled': bool(raw[0]),
        'interlock_enable_mask': int(raw[1]),
        'fan_switch_override': int(raw[2]),
        'heat': {
            'limit_voltage_v': heater_limit(3),
            'limit_current_a': heater_limit(5),
            'limit_power_w': heater_power(7),
            'target_temperature_c': milli_units(11) - 273.15,
            'enabled': bool(raw[15]),
        },
        'hv': {},
    }
    for address in (1, 2, 3):
        offset = HV_CONFIG_BASE_OFFSET + (address - 1) * HV_CONFIG_STRIDE
        decoded['hv'][address] = {
            'target_voltage_v': milli_units(offset),
            'max_voltage_step_v': milli_units(offset + 4),
            'negative_adc': bool(raw[offset + 8]),
            'high_current_range': bool(raw[offset + 9]),
            'enabled': bool(raw[offset + 10]),
        }
    return decoded

def read_current_configuration(device):
    properties = record_call(
        device, 'get_config_values', device.get_config_values, required=True
    )
    raw = record_call(
        device, 'get_current_config', device.get_current_config, required=True
    )
    values = raw.get('values', [])
    raw_config = values[0] if values else []
    return {
        'properties': properties,
        'raw': raw,
        'raw_bytes': list(raw_config),
        'decoded': decode_current_config(raw_config),
    }

def safe_config_violations(decoded):
    violations = []
    if decoded['device_enabled']:
        violations.append('global device gate is enabled')
    heat = decoded['heat']
    if heat['enabled']:
        violations.append('heater is enabled')
    if not math.isclose(heat['target_temperature_c'], 0.0, abs_tol=1e-6):
        violations.append(
            f"heater target is {heat['target_temperature_c']} degC"
        )
    for address, hv in decoded['hv'].items():
        if hv['enabled']:
            violations.append(f'HV{address} module gate is enabled')
        if not math.isclose(hv['target_voltage_v'], 0.0, abs_tol=1e-6):
            violations.append(
                f"HV{address} target is {hv['target_voltage_v']} V"
            )
    return violations

def build_temporary_hv_config(raw_config, address, max_voltage_step):
    raw = bytearray(raw_config)
    if len(raw) != ESIBase.CONFIG_DATA_SIZE:
        raise RuntimeError('Cannot build a temporary configuration of the wrong size.')
    selected_offset = (
        HV_CONFIG_BASE_OFFSET + (address - 1) * HV_CONFIG_STRIDE
    )
    max_step_raw = int(round(float(max_voltage_step) * 1000.0))
    if max_step_raw != MAX_VOLTAGE_STEP_RAW:
        raise RuntimeError('Unexpected MaxVoltStep fixed-point value.')
    struct.pack_into(
        '<i', raw, selected_offset + HV_MAX_STEP_RELATIVE_OFFSET, max_step_raw
    )
    return list(raw)

def temporary_config_violations(decoded, address, max_voltage_step):
    violations = safe_config_violations(decoded)
    selected = decoded['hv'][address]
    if not math.isclose(
        selected['max_voltage_step_v'], float(max_voltage_step),
        rel_tol=1e-6, abs_tol=1e-6,
    ):
        violations.append(
            'selected HV max-voltage step was not applied: '
            f"{selected['max_voltage_step_v']} V"
        )
    return violations

print('Driver loaded. No COM port has been opened yet.')


## Run the activation probe

At `0 V`, the module can remain yellow even when both activation readbacks are true: the shipped utility reports the same red+green LED state at a zero target. The July DLL must return status `0`; require both `direct_activation` and the independent `pwm_activation` readback to match the requested state before considering a nonzero test.

Use `ARM_TEMP_CONFIG=True, ARM_NONZERO_TEST=False` first. This validates configuration application and automatic restoration while every voltage target remains zero. Only a successful JSON report with an exact safe restoration permits a later run with both switches true.


In [ ]:
def run_activation_probe():
    if MODULE_ADDRESS not in (1, 2):
        raise ValueError('MODULE_ADDRESS must be 1 or 2.')
    if not 0.0 <= float(TEST_VOLTAGE) <= 3000.0:
        raise ValueError('TEST_VOLTAGE must be between 0 and 3000 V.')
    if ARM_NONZERO_TEST and not ARM_TEMP_CONFIG:
        raise ValueError(
            'ARM_NONZERO_TEST requires ARM_TEMP_CONFIG so the HV control '
            'parameter is configured and verified first.'
        )
    if ARM_NONZERO_TEST and not math.isclose(
        float(TEST_VOLTAGE), 100.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep TEST_VOLTAGE at 100 V for this guarded probe.')
    if not math.isclose(
        float(NONZERO_RISE_TIMEOUT_SECONDS), 10.0,
        rel_tol=0.0, abs_tol=1e-9,
    ):
        raise ValueError('Keep NONZERO_RISE_TIMEOUT_SECONDS at 10 seconds.')
    if not math.isclose(
        float(NONZERO_HOLD_SECONDS), 20.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep NONZERO_HOLD_SECONDS at 20 seconds.')
    if not math.isclose(
        float(NONZERO_HOLD_START_V), 95.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep NONZERO_HOLD_START_V at 95 V.')
    if not math.isclose(
        float(NONZERO_POLL_SECONDS), 0.05, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep NONZERO_POLL_SECONDS at 0.05 seconds.')
    if not math.isclose(
        float(NONZERO_ABS_LIMIT_V), 150.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep NONZERO_ABS_LIMIT_V at 150 V.')
    if not math.isclose(
        float(NONZERO_ADC_GRACE_SECONDS), 1.5, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep NONZERO_ADC_GRACE_SECONDS at 1.5 seconds.')
    if not math.isclose(
        float(DISCHARGE_LIMIT_V), 1.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep DISCHARGE_LIMIT_V at 1 V.')
    if not math.isclose(
        float(DISCHARGE_TIMEOUT_SECONDS), 60.0, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep DISCHARGE_TIMEOUT_SECONDS at 60 seconds.')
    if not math.isclose(
        float(DISCHARGE_POLL_SECONDS), 0.25, rel_tol=0.0, abs_tol=1e-9
    ):
        raise ValueError('Keep DISCHARGE_POLL_SECONDS at 0.25 seconds.')
    if not math.isclose(float(MAX_VOLTAGE_STEP), 10.008, abs_tol=1e-6):
        raise ValueError('Keep MAX_VOLTAGE_STEP at the vendor value 10.008 V.')
    if int(MAX_VOLTAGE_STEP_RAW) != 10008:
        raise ValueError('Keep MAX_VOLTAGE_STEP_RAW at 10008 millivolts.')

    device = ESIBase(
        com=COM_PORT,
        idn='esi_hv_activation_probe',
        dll_path=DLL_FILE,
        error_codes_path=ERROR_CODES_FILE,
    )
    opened = False
    initial_measurement_ranges = None
    initial_config_raw = None
    initial_config_safe = False
    safety_sequence_started = False
    temporary_config_attempted = False
    report = {
        'timestamp': datetime.now().astimezone().isoformat(),
        'com_port': COM_PORT,
        'module_address': MODULE_ADDRESS,
        'dll_sha256': DLL_SHA256,
        'dll_sw_version': int(device.get_sw_version()),
        'armed_temporary_config': bool(ARM_TEMP_CONFIG),
        'armed_nonzero': bool(ARM_NONZERO_TEST),
        'test_voltage_v': float(TEST_VOLTAGE),
        'nonzero_rise_timeout_seconds': float(NONZERO_RISE_TIMEOUT_SECONDS),
        'nonzero_hold_seconds': float(NONZERO_HOLD_SECONDS),
        'nonzero_hold_start_v': float(NONZERO_HOLD_START_V),
        'nonzero_poll_seconds': float(NONZERO_POLL_SECONDS),
        'nonzero_abs_limit_v': float(NONZERO_ABS_LIMIT_V),
        'nonzero_adc_grace_seconds': float(NONZERO_ADC_GRACE_SECONDS),
        'discharge_limit_v': float(DISCHARGE_LIMIT_V),
        'discharge_timeout_seconds': float(DISCHARGE_TIMEOUT_SECONDS),
        'nominal_evaluation': 'measurement_only',
        'max_voltage_step_v': float(MAX_VOLTAGE_STEP),
        'max_voltage_step_raw': int(MAX_VOLTAGE_STEP_RAW),
        'steps': {},
        'cleanup': {},
    }

    def wait_for_discharge():
        discharge_started = time.monotonic()
        discharge_samples = []
        discharge_confirmed = False
        while True:
            discharge_pwm = record_call(
                device, 'discharge_get_pwm',
                device.get_hv_supply_params_pwm, MODULE_ADDRESS,
            )
            discharge_adc = record_call(
                device, 'discharge_get_voltage',
                device.get_hv_supply_output_voltage, MODULE_ADDRESS,
            )
            elapsed = time.monotonic() - discharge_started
            pwm_values = discharge_pwm.get('values', [])
            adc_values = discharge_adc.get('values', [])
            pwm_measured_v = (
                float(pwm_values[5]) if len(pwm_values) >= 6 else None
            )
            adc_valid = len(adc_values) >= 2 and adc_values[0] is True
            adc_v = float(adc_values[1]) if adc_valid else None
            sample = {
                'elapsed_s': round(elapsed, 6),
                'pwm': discharge_pwm,
                'measured_voltage': discharge_adc,
                'pwm_measured_v': pwm_measured_v,
                'adc_valid': adc_valid,
                'adc_v': adc_v,
            }
            discharge_samples.append(sample)
            if (
                pwm_measured_v is not None
                and math.isfinite(pwm_measured_v)
                and abs(pwm_measured_v) <= DISCHARGE_LIMIT_V
                and adc_valid
                and math.isfinite(adc_v)
                and abs(adc_v) <= DISCHARGE_LIMIT_V
            ):
                discharge_confirmed = True
                break
            if elapsed >= DISCHARGE_TIMEOUT_SECONDS:
                break
            time.sleep(DISCHARGE_POLL_SECONDS)
        return {
            'confirmed': discharge_confirmed,
            'limit_v': float(DISCHARGE_LIMIT_V),
            'timeout_s': float(DISCHARGE_TIMEOUT_SECONDS),
            'samples': discharge_samples,
        }
    try:
        report['steps']['open'] = record_call(
            device, 'open_port', device.open_port, COM_PORT, required=True
        )
        opened = True
        report['steps']['baud'] = record_call(
            device, 'set_comspeed', device.set_comspeed, BAUD_RATE, required=True
        )
        initial_ranges = record_call(
            device, 'get_initial_measurement_ranges',
            device.get_hv_supply_meas_ranges, MODULE_ADDRESS, required=True,
        )
        report['steps']['initial_measurement_ranges'] = initial_ranges
        initial_measurement_ranges = tuple(
            bool(value) for value in initial_ranges['values'][:2]
        )
        initial_config = read_current_configuration(device)
        initial_config_raw = initial_config['raw_bytes']
        initial_violations = safe_config_violations(initial_config['decoded'])
        initial_config_safe = not initial_violations
        initial_config['safe_for_automatic_restore'] = initial_config_safe
        initial_config['safety_violations'] = initial_violations
        report['steps']['initial_current_configuration'] = initial_config
        if ARM_TEMP_CONFIG and not initial_config_safe:
            raise RuntimeError(
                'Refusing temporary configuration because the initial '
                f"configuration is not safely restorable: {initial_violations}"
            )

        safety_sequence_started = True
        report['steps']['global_off_initial'] = record_call(
            device, 'set_enable(False)', device.set_enable, False, required=True
        )
        for address in (1, 2):
            report['steps'][f'zero_target_{address}'] = record_call(
                device, f'zero_target_{address}',
                device.set_hv_supply_target_output_voltage, address, 0.0,
                required=True,
            )
            report['steps'][f'initial_standby_{address}'] = record_call(
                device, f'initial_standby_{address}',
                device.set_module_activation_state, address, False,
                required=True,
            )
        report['steps']['zero_heat'] = record_call(
            device, 'zero_heat', device.set_heat_ctrl_heater_temperature, 0.0
        )
        time.sleep(SETTLE_SECONDS)
        pre_config_safe_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['pre_config_safe_state'] = pre_config_safe_state
        if (
            pre_config_safe_state['global_enabled'] is not False
            or pre_config_safe_state['target_v'] != 0.0
            or pre_config_safe_state['direct_activation'] is not False
            or pre_config_safe_state['pwm_activation'] is not False
        ):
            raise RuntimeError(
                'Refusing configuration write because safe hardware OFF '
                'was not confirmed.'
            )
        if ARM_TEMP_CONFIG:
            temporary_config = build_temporary_hv_config(
                initial_config_raw, MODULE_ADDRESS, MAX_VOLTAGE_STEP
            )
            max_step_offset = (
                HV_CONFIG_BASE_OFFSET
                + (MODULE_ADDRESS - 1) * HV_CONFIG_STRIDE
                + HV_MAX_STEP_RELATIVE_OFFSET
            )
            allowed_offsets = set(range(max_step_offset, max_step_offset + 4))
            changed_offsets = [
                index for index, (before, after) in enumerate(
                    zip(initial_config_raw, temporary_config, strict=True)
                ) if before != after
            ]
            report['steps']['temporary_configuration_patch'] = {
                'allowed_offsets': sorted(allowed_offsets),
                'changed_offsets': changed_offsets,
            }
            if not changed_offsets or not set(changed_offsets) <= allowed_offsets:
                raise RuntimeError(
                    'Temporary configuration attempted to change bytes outside '
                    'the selected MaxVoltStep field.'
                )
            temporary_config_attempted = True
            report['steps']['set_temporary_configuration'] = record_call(
                device, 'set_temporary_configuration',
                device.set_current_config, temporary_config,
            )
            set_config_status = report['steps'][
                'set_temporary_configuration'
            ].get('status')
            if set_config_status not in (device.NO_ERR, device.ERR_DATA_RECEIVE):
                raise RuntimeError(
                    'Temporary configuration write failed before readback: '
                    f"{report['steps']['set_temporary_configuration']}"
                )
            time.sleep(SETTLE_SECONDS)
            temporary_readback = read_current_configuration(device)
            temporary_readback['write_status'] = set_config_status
            temporary_readback['write_acknowledged'] = (
                set_config_status == device.NO_ERR
            )
            temporary_readback['matches_requested_bytes'] = (
                temporary_readback['raw_bytes'] == temporary_config
            )
            temporary_violations = temporary_config_violations(
                temporary_readback['decoded'], MODULE_ADDRESS, MAX_VOLTAGE_STEP
            )
            readback_changed_offsets = [
                index for index, (before, after) in enumerate(
                    zip(
                        initial_config_raw, temporary_readback['raw_bytes'],
                        strict=True,
                    )
                ) if before != after
            ]
            temporary_readback['changed_offsets_from_initial'] = (
                readback_changed_offsets
            )
            if not set(readback_changed_offsets) <= allowed_offsets:
                temporary_violations.append(
                    'Controller changed configuration bytes outside MaxVoltStep: '
                    f'{readback_changed_offsets}'
                )
            if not temporary_readback['matches_requested_bytes']:
                temporary_violations.append(
                    'Configuration readback does not exactly match the requested bytes.'
                )
            temporary_readback['verification_violations'] = temporary_violations
            report['steps']['temporary_configuration_readback'] = (
                temporary_readback
            )
            if temporary_violations:
                raise RuntimeError(
                    'Temporary configuration verification failed: '
                    f'{temporary_violations}'
                )
        time.sleep(SETTLE_SECONDS)
        baseline_global_off = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['baseline_global_off'] = baseline_global_off
        if ARM_TEMP_CONFIG and (
            baseline_global_off['global_enabled'] is not False
            or baseline_global_off['target_v'] != 0.0
            or baseline_global_off['direct_activation'] is not False
            or baseline_global_off['pwm_activation'] is not False
        ):
            raise RuntimeError(
                'Temporary zero-target hardware state was not confirmed.'
            )
        standby_off_request = record_call(
            device, 'set_module_activation_state(False) with global OFF',
            device.set_module_activation_state, MODULE_ADDRESS, False, required=True
        )
        report['steps']['request_standby_global_off'] = standby_off_request
        time.sleep(SETTLE_SECONDS)
        standby_off_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_global_off_first'] = standby_off_state
        if (
            standby_off_request.get('status') != device.NO_ERR
            or standby_off_state['direct_activation'] is not False
            or standby_off_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_global_off_retry'] = record_call(
                device, 'retry set_module_activation_state(False) with global OFF',
                device.set_module_activation_state, MODULE_ADDRESS, False, required=True
            )
            time.sleep(SETTLE_SECONDS)
            standby_off_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_global_off_retry'] = standby_off_state
        report['steps']['standby_global_off_final'] = standby_off_state
        if (
            standby_off_state['direct_activation'] is not False
            or standby_off_state['pwm_activation'] is not False
        ):
            raise RuntimeError('Could not confirm standby with global gate OFF.')
        first_off_request = record_call(
            device, 'set_module_activation_state(True) with global OFF',
            device.set_module_activation_state, MODULE_ADDRESS, True, required=True
        )
        report['steps']['request_active_global_off'] = first_off_request
        time.sleep(SETTLE_SECONDS)
        first_off_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['active_global_off_first'] = first_off_state
        off_active_state = first_off_state
        if (
            first_off_request.get('status') != device.NO_ERR
            or first_off_state['direct_activation'] is not True
            or first_off_state['pwm_activation'] is not True
        ):
            report['steps']['request_active_global_off_retry'] = record_call(
                device, 'retry set_module_activation_state(True) with global OFF',
                device.set_module_activation_state, MODULE_ADDRESS, True, required=True
            )
            time.sleep(SETTLE_SECONDS)
            off_active_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['active_global_off_retry'] = off_active_state
        report['steps']['active_global_off_final'] = off_active_state

        reset_request = record_call(
            device, 'set_module_activation_state(False) before global ON',
            device.set_module_activation_state, MODULE_ADDRESS, False, required=True
        )
        report['steps']['request_standby_reset'] = reset_request
        time.sleep(SETTLE_SECONDS)
        reset_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_reset_global_off_first'] = reset_state
        if (
            reset_request.get('status') != device.NO_ERR
            or reset_state['direct_activation'] is not False
            or reset_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_reset_retry'] = record_call(
                device, 'retry set_module_activation_state(False) before global ON',
                device.set_module_activation_state, MODULE_ADDRESS, False, required=True
            )
            time.sleep(SETTLE_SECONDS)
            reset_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_reset_global_off_retry'] = reset_state
        report['steps']['standby_reset_global_off_final'] = reset_state
        if (
            reset_state['direct_activation'] is not False
            or reset_state['pwm_activation'] is not False
        ):
            raise RuntimeError('Could not confirm standby before global-ON test.')
        report['steps']['global_on_for_test'] = record_call(
            device, 'set_enable(True)', device.set_enable, True, required=True
        )
        time.sleep(SETTLE_SECONDS)
        report['steps']['baseline_global_on'] = read_hv_state(
            device, MODULE_ADDRESS
        )
        standby_on_request = record_call(
            device, 'set_module_activation_state(False) with global ON',
            device.set_module_activation_state, MODULE_ADDRESS, False, required=True
        )
        report['steps']['request_standby_global_on'] = standby_on_request
        time.sleep(SETTLE_SECONDS)
        standby_on_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['standby_global_on_first'] = standby_on_state
        if (
            standby_on_request.get('status') != device.NO_ERR
            or standby_on_state['direct_activation'] is not False
            or standby_on_state['pwm_activation'] is not False
        ):
            report['steps']['request_standby_global_on_retry'] = record_call(
                device, 'retry set_module_activation_state(False) with global ON',
                device.set_module_activation_state, MODULE_ADDRESS, False, required=True
            )
            time.sleep(SETTLE_SECONDS)
            standby_on_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['standby_global_on_retry'] = standby_on_state
        report['steps']['standby_global_on_final'] = standby_on_state
        if (
            standby_on_state['direct_activation'] is not False
            or standby_on_state['pwm_activation'] is not False
        ):
            raise RuntimeError('Could not confirm standby with global gate ON.')
        input('Observe the module LED in STANDBY with global ON, then press Enter...')

        first_request = record_call(
            device, 'set_module_activation_state(True) with global ON',
            device.set_module_activation_state, MODULE_ADDRESS, True, required=True
        )
        report['steps']['request_active_global_on'] = first_request
        time.sleep(SETTLE_SECONDS)
        first_active_state = read_hv_state(device, MODULE_ADDRESS)
        report['steps']['active_global_on_first'] = first_active_state
        active_state = first_active_state
        if (
            first_request.get('status') != device.NO_ERR
            or first_active_state['direct_activation'] is not True
            or first_active_state['pwm_activation'] is not True
        ):
            report['steps']['request_active_global_on_retry'] = record_call(
                device, 'retry set_module_activation_state(True) with global ON',
                device.set_module_activation_state, MODULE_ADDRESS, True, required=True
            )
            time.sleep(SETTLE_SECONDS)
            active_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['active_global_on_retry'] = active_state
        report['steps']['active_zero_v_final'] = active_state
        zero_gate_violations = []
        expected_zero_gate_state = {
            'global_enabled': True,
            'direct_activation': True,
            'pwm_activation': True,
            'module_gate_active': True,
            'device_gate_active': True,
        }
        for name, expected in expected_zero_gate_state.items():
            if active_state.get(name) is not expected:
                zero_gate_violations.append(
                    f'{name}: expected {expected}, got {active_state.get(name)}'
                )
        for name in ('target_v', 'pwm_voltage_set_v', 'pwm_width_s'):
            value = active_state.get(name)
            if value is None or not math.isclose(
                float(value), 0.0, rel_tol=0.0, abs_tol=1e-9
            ):
                zero_gate_violations.append(f'{name}: expected 0, got {value}')
        if zero_gate_violations:
            raise RuntimeError(
                f'Zero-target PWM gate verification failed: {zero_gate_violations}'
            )

        current_high = initial_measurement_ranges[1]
        zero_adc_results = {}
        for polarity, negative in (('positive', False), ('negative', True)):
            result = select_and_read_adc(
                device, MODULE_ADDRESS, negative, current_high
            )
            report['steps'][f'zero_{polarity}_adc'] = result
            measured = result['state']['measured_voltage'].get('values', [])
            if len(measured) < 2 or measured[0] is not True:
                raise RuntimeError(
                    f'{polarity} ADC did not return a valid zero-target reading: '
                    f'{measured}'
                )
            measured_v = float(measured[1])
            zero_adc_results[polarity] = measured_v
            if (
                not math.isfinite(measured_v)
                or abs(measured_v) > ZERO_ADC_ABS_LIMIT_V
            ):
                raise RuntimeError(
                    f'{polarity} ADC is not near zero: {measured_v} V '
                    f'(limit +/-{ZERO_ADC_ABS_LIMIT_V:g} V)'
                )
        report['steps']['restore_measurement_ranges_after_zero_adc'] = record_call(
            device, 'restore_measurement_ranges_after_zero_adc',
            device.set_hv_supply_meas_ranges, MODULE_ADDRESS,
            *initial_measurement_ranges, required=True,
        )
        time.sleep(SETTLE_SECONDS)
        restored_zero_ranges = record_call(
            device, 'verify_measurement_ranges_after_zero_adc',
            device.get_hv_supply_meas_ranges, MODULE_ADDRESS, required=True,
        )
        report['steps']['verify_measurement_ranges_after_zero_adc'] = (
            restored_zero_ranges
        )
        if restored_zero_ranges.get('values', [])[:2] != list(
            initial_measurement_ranges
        ):
            raise RuntimeError(
                'Initial ADC selection was not restored after the zero-target '
                f"checks: {restored_zero_ranges.get('values', [])[:2]}"
            )
        active_max_step_v = None
        if ARM_TEMP_CONFIG:
            zero_active_config = read_current_configuration(device)
            selected_config = zero_active_config['decoded']['hv'][MODULE_ADDRESS]
            active_max_step_v = selected_config['max_voltage_step_v']
            active_max_step_offset = (
                HV_CONFIG_BASE_OFFSET
                + (MODULE_ADDRESS - 1) * HV_CONFIG_STRIDE
                + HV_MAX_STEP_RELATIVE_OFFSET
            )
            active_max_step_bytes = zero_active_config['raw_bytes'][
                active_max_step_offset:active_max_step_offset + 4
            ]
            zero_active_config['selected_max_step_bytes'] = active_max_step_bytes
            report['steps']['zero_active_current_configuration'] = (
                zero_active_config
            )
            if (
                selected_config['target_voltage_v'] != 0.0
                or not math.isclose(
                    active_max_step_v, float(MAX_VOLTAGE_STEP),
                    rel_tol=0.0, abs_tol=1e-6,
                )
                or active_max_step_bytes
                != list(struct.pack('<i', MAX_VOLTAGE_STEP_RAW))
            ):
                raise RuntimeError(
                    'Corrected MaxVoltStep layout did not survive the active '
                    f'zero-target sequence: {selected_config}, '
                    f'bytes={active_max_step_bytes}'
                )
        report['zero_validation_summary'] = {
            'global_enabled': active_state['global_enabled'],
            'direct_activation': active_state['direct_activation'],
            'pwm_activation': active_state['pwm_activation'],
            'module_state_raw': active_state['module_state_raw'],
            'control_active': active_state['control_active'],
            'target_v': active_state['target_v'],
            'pwm_voltage_set_v': active_state['pwm_voltage_set_v'],
            'positive_adc_v': zero_adc_results['positive'],
            'negative_adc_v': zero_adc_results['negative'],
            'adc_abs_limit_v': float(ZERO_ADC_ABS_LIMIT_V),
            'initial_measurement_ranges_restored': True,
            'active_max_voltage_step_v': active_max_step_v,
        }
        input('Observe the module LED after ACTIVE requests at 0 V, then press Enter...')

        if ARM_NONZERO_TEST:
            if active_state['direct_activation'] is not True:
                raise RuntimeError('Direct status did not confirm module activation.')
            if active_state['pwm_activation'] is not True:
                raise RuntimeError('PWM status did not confirm module activation.')
            if active_state['global_enabled'] is not True:
                raise RuntimeError('Global enable readback is not ON.')
            expected_confirmation = f'ARM {TEST_VOLTAGE:g} V'
            operator_confirmation = input(
                'Confirm both external meters are already connected, nobody '
                'will touch the energized setup, and the area is clear. Type '
                f'{expected_confirmation!r} to continue: '
            ).strip()
            report['nonzero_operator_confirmation'] = operator_confirmation
            if operator_confirmation != expected_confirmation:
                raise RuntimeError('The guarded operator confirmation did not match.')

            def abort_nonzero(reason):
                report['nonzero_guard_abort_reason'] = reason
                report['steps']['nonzero_guard_global_off'] = record_call(
                    device, 'nonzero_guard_global_off',
                    device.set_enable, False, required=True,
                )
                report['steps']['nonzero_guard_zero_target'] = record_call(
                    device, 'nonzero_guard_zero_target',
                    device.set_hv_supply_target_output_voltage,
                    MODULE_ADDRESS, 0.0, required=True,
                )
                report['steps']['nonzero_guard_standby'] = record_call(
                    device, 'nonzero_guard_standby',
                    device.set_module_activation_state,
                    MODULE_ADDRESS, False, required=True,
                )
                raise RuntimeError(f'Guarded nonzero probe aborted: {reason}')

            def validate_nonzero_adc(polarity, adc_result):
                adc_state = adc_result['state']
                adc_values = adc_state['measured_voltage'].get('values', [])
                adc_pwm_v = adc_state.get('pwm_voltage_measured_v')
                if len(adc_values) < 2 or adc_values[0] is not True:
                    abort_nonzero(f'{polarity} ADC became invalid: {adc_values}')
                if adc_pwm_v is None:
                    abort_nonzero(f'{polarity} PWM readback became invalid')
                adc_max = max(abs(float(adc_values[1])), abs(float(adc_pwm_v)))
                for settle_sample in adc_result.get('settle_guard_samples', []):
                    settle_pwm_v = settle_sample.get('pwm_measured_v')
                    if settle_pwm_v is not None:
                        adc_max = max(adc_max, abs(float(settle_pwm_v)))
                if not math.isfinite(adc_max) or adc_max > NONZERO_ABS_LIMIT_V:
                    abort_nonzero(
                        f'{polarity} ADC/PWM magnitude {adc_max:g} V exceeds '
                        f'{NONZERO_ABS_LIMIT_V:g} V'
                    )
                return adc_max

            report['steps']['nonzero_prepare_standby'] = record_call(
                device, 'nonzero_prepare_standby',
                device.set_module_activation_state,
                MODULE_ADDRESS, False, required=True,
            )
            time.sleep(SETTLE_SECONDS)
            staged_off_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['nonzero_staged_off_state'] = staged_off_state
            if (
                staged_off_state['global_enabled'] is not True
                or staged_off_state['direct_activation'] is not False
                or staged_off_state['pwm_activation'] is not False
                or staged_off_state['target_v'] != 0.0
            ):
                abort_nonzero(
                    f'could not stage target-first activation: {staged_off_state}'
                )

            print(
                f'Staging {TEST_VOLTAGE:g} V while the module is OFF. '
                f'Rise timeout: {NONZERO_RISE_TIMEOUT_SECONDS:g} s; '
                f'hold: {NONZERO_HOLD_SECONDS:g} s after PWM measured '
                f'reaches {NONZERO_HOLD_START_V:g} V. '
                'Do not touch the setup.',
                flush=True,
            )
            report['steps']['set_test_voltage'] = record_call(
                device, 'set_test_voltage',
                device.set_hv_supply_target_output_voltage,
                MODULE_ADDRESS, float(TEST_VOLTAGE), required=True,
            )
            staged_target_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['nonzero_staged_target_state'] = staged_target_state
            staged_config = read_current_configuration(device)
            report['steps']['nonzero_staged_configuration'] = staged_config
            staged_selected_config = staged_config['decoded']['hv'][MODULE_ADDRESS]
            if (
                staged_target_state['global_enabled'] is not True
                or staged_target_state['direct_activation'] is not False
                or staged_target_state['pwm_activation'] is not False
                or not math.isclose(
                    staged_target_state['target_v'], float(TEST_VOLTAGE),
                    rel_tol=0.0, abs_tol=1e-6,
                )
                or not math.isclose(
                    staged_selected_config['target_voltage_v'],
                    float(TEST_VOLTAGE), rel_tol=0.0, abs_tol=1e-6,
                )
                or not math.isclose(
                    staged_selected_config['max_voltage_step_v'],
                    float(MAX_VOLTAGE_STEP), rel_tol=0.0, abs_tol=1e-6,
                )
            ):
                abort_nonzero(
                    f'staged target/configuration verification failed: '
                    f'{staged_target_state}, {staged_selected_config}'
                )
            report['steps']['nonzero_activate_after_target'] = record_call(
                device, 'nonzero_activate_after_target',
                device.set_module_activation_state,
                MODULE_ADDRESS, True, required=True,
            )
            report['steps']['nonzero_reassert_global_after_target'] = record_call(
                device, 'nonzero_reassert_global_after_target',
                device.set_enable, True, required=True,
            )
            guard_started = time.monotonic()
            guard_samples = []
            guard_max_abs_voltage_v = 0.0
            last_valid_adc_elapsed = None
            last_valid_adc_v = None
            hold_started_elapsed = None
            hold_elapsed_seconds = 0.0
            while True:
                sample = {
                    'elapsed_s': round(time.monotonic() - guard_started, 6),
                    'target': record_call(
                        device, 'guard_get_target',
                        device.get_hv_supply_target_output_voltage,
                        MODULE_ADDRESS, required=True,
                    ),
                    'module_state': record_call(
                        device, 'guard_get_module_state',
                        device.get_module_state, MODULE_ADDRESS, required=True,
                    ),
                    'pwm': record_call(
                        device, 'guard_get_pwm',
                        device.get_hv_supply_params_pwm,
                        MODULE_ADDRESS, required=True,
                    ),
                    'measured_voltage': record_call(
                        device, 'guard_get_measured_voltage',
                        device.get_hv_supply_output_voltage,
                        MODULE_ADDRESS, required=True,
                    ),
                }
                target_values = sample['target'].get('values', [])
                module_state_values = sample['module_state'].get('values', [])
                pwm_values = sample['pwm'].get('values', [])
                voltage_values = sample['measured_voltage'].get('values', [])
                sample_elapsed = time.monotonic() - guard_started
                sample['elapsed_s'] = round(sample_elapsed, 6)
                guard_reason = None
                if (
                    not target_values
                    or not math.isclose(
                        float(target_values[0]), float(TEST_VOLTAGE),
                        rel_tol=0.0, abs_tol=1e-6,
                    )
                ):
                    guard_reason = f'target readback changed: {target_values}'
                elif not module_state_values:
                    guard_reason = 'module state readback is missing'
                elif (
                    int(module_state_values[0]) & int(device.MS_ACTIVE)
                ) != int(device.MS_ACTIVE):
                    guard_reason = (
                        f'active gate/control bits were lost: '
                        f'{int(module_state_values[0]):#06x}'
                    )
                elif len(pwm_values) < 7:
                    guard_reason = f'incomplete PWM readback: {pwm_values}'
                elif not bool(pwm_values[6]):
                    guard_reason = 'PWM activation readback became false'
                else:
                    pwm_measured_v = float(pwm_values[5])
                    sample['pwm_measured_v'] = pwm_measured_v
                    sample['data_ready_flags'] = (
                        int(pwm_values[7]) if len(pwm_values) >= 8 else None
                    )
                    adc_valid = (
                        len(voltage_values) >= 2 and voltage_values[0] is True
                    )
                    sample['adc_valid'] = adc_valid
                    sample_max = abs(pwm_measured_v)
                    if adc_valid:
                        adc_v = float(voltage_values[1])
                        sample['adc_v'] = adc_v
                        last_valid_adc_elapsed = sample_elapsed
                        last_valid_adc_v = adc_v
                        sample_max = max(sample_max, abs(adc_v))
                    else:
                        adc_silence_seconds = (
                            sample_elapsed
                            if last_valid_adc_elapsed is None
                            else sample_elapsed - last_valid_adc_elapsed
                        )
                        sample['adc_silence_seconds'] = adc_silence_seconds
                        sample['last_valid_adc_v'] = last_valid_adc_v
                        if adc_silence_seconds > NONZERO_ADC_GRACE_SECONDS:
                            guard_reason = (
                                f'No fresh valid ADC conversion for '
                                f'{adc_silence_seconds:.3f} s: {voltage_values}'
                            )
                    sample['max_abs_voltage_v'] = sample_max
                    guard_max_abs_voltage_v = max(
                        guard_max_abs_voltage_v, sample_max
                    )
                    if guard_reason is None and (
                        not math.isfinite(sample_max)
                        or sample_max > NONZERO_ABS_LIMIT_V
                    ):
                        guard_reason = (
                            f'PWM/ADC magnitude {sample_max:g} V exceeds '
                            f'{NONZERO_ABS_LIMIT_V:g} V'
                        )
                    if guard_reason is None and hold_started_elapsed is None:
                        sample['phase'] = 'rise'
                        if abs(pwm_measured_v) >= NONZERO_HOLD_START_V:
                            hold_started_elapsed = sample_elapsed
                            sample['phase'] = 'hold'
                            sample['hold_started'] = True
                            print(
                                f'PWM measured reached {abs(pwm_measured_v):g} V; '
                                f'starting {NONZERO_HOLD_SECONDS:g} s hold.',
                                flush=True,
                            )
                        elif sample_elapsed >= NONZERO_RISE_TIMEOUT_SECONDS:
                            guard_reason = (
                                f'PWM measured did not reach '
                                f'{NONZERO_HOLD_START_V:g} V within '
                                f'{NONZERO_RISE_TIMEOUT_SECONDS:g} s'
                            )
                    if hold_started_elapsed is not None:
                        hold_elapsed_seconds = (
                            sample_elapsed - hold_started_elapsed
                        )
                        sample['phase'] = 'hold'
                        sample['hold_elapsed_s'] = hold_elapsed_seconds
                guard_samples.append(sample)
                if guard_reason is not None:
                    report['steps']['nonzero_guard_samples'] = guard_samples
                    report['nonzero_hold_summary'] = {
                        'start_threshold_v': float(NONZERO_HOLD_START_V),
                        'started_elapsed_s': hold_started_elapsed,
                        'elapsed_s': hold_elapsed_seconds,
                        'requested_duration_s': float(NONZERO_HOLD_SECONDS),
                        'completed': False,
                    }
                    abort_nonzero(guard_reason)
                if (
                    hold_started_elapsed is not None
                    and hold_elapsed_seconds >= NONZERO_HOLD_SECONDS
                ):
                    break
                remaining = (
                    NONZERO_RISE_TIMEOUT_SECONDS - sample_elapsed
                    if hold_started_elapsed is None
                    else NONZERO_HOLD_SECONDS - hold_elapsed_seconds
                )
                time.sleep(
                    max(0.0, min(NONZERO_POLL_SECONDS, remaining))
                )
            report['steps']['nonzero_guard_samples'] = guard_samples
            report['nonzero_hold_summary'] = {
                'start_threshold_v': float(NONZERO_HOLD_START_V),
                'started_elapsed_s': hold_started_elapsed,
                'elapsed_s': hold_elapsed_seconds,
                'requested_duration_s': float(NONZERO_HOLD_SECONDS),
                'completed': True,
            }
            nonzero_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['nonzero_before_adc_selection'] = nonzero_state
            nonzero_voltage_values = nonzero_state['measured_voltage'].get(
                'values', []
            )
            if nonzero_state['pwm_voltage_measured_v'] is None:
                abort_nonzero(
                    'full-state PWM voltage readback became invalid'
                )
            full_state_max = abs(float(nonzero_state['pwm_voltage_measured_v']))
            if (
                len(nonzero_voltage_values) >= 2
                and nonzero_voltage_values[0] is True
            ):
                full_state_max = max(
                    full_state_max, abs(float(nonzero_voltage_values[1]))
                )
            guard_max_abs_voltage_v = max(guard_max_abs_voltage_v, full_state_max)
            if (
                not math.isfinite(full_state_max)
                or full_state_max > NONZERO_ABS_LIMIT_V
            ):
                abort_nonzero(
                    f'full-state ADC/PWM magnitude {full_state_max:g} V exceeds '
                    f'{NONZERO_ABS_LIMIT_V:g} V'
                )
            nonzero_config = read_current_configuration(device)
            report['steps']['nonzero_current_configuration'] = nonzero_config
            nonzero_selected_config = nonzero_config['decoded']['hv'][
                MODULE_ADDRESS
            ]
            nonzero_max_step_offset = (
                HV_CONFIG_BASE_OFFSET
                + (MODULE_ADDRESS - 1) * HV_CONFIG_STRIDE
                + HV_MAX_STEP_RELATIVE_OFFSET
            )
            nonzero_max_step_bytes = nonzero_config['raw_bytes'][
                nonzero_max_step_offset:nonzero_max_step_offset + 4
            ]
            nonzero_config['selected_max_step_bytes'] = nonzero_max_step_bytes
            if (
                not math.isclose(
                    nonzero_selected_config['target_voltage_v'],
                    float(TEST_VOLTAGE), rel_tol=0.0, abs_tol=1e-6,
                )
                or not math.isclose(
                    nonzero_selected_config['max_voltage_step_v'],
                    float(MAX_VOLTAGE_STEP), rel_tol=0.0, abs_tol=1e-6,
                )
                or nonzero_max_step_bytes
                != list(struct.pack('<i', MAX_VOLTAGE_STEP_RAW))
            ):
                abort_nonzero(
                    f'nonzero configuration changed unexpectedly: '
                    f'{nonzero_selected_config}, bytes={nonzero_max_step_bytes}'
                )
            if (
                nonzero_state['target_v'] is None
                or not math.isclose(
                    nonzero_state['target_v'], float(TEST_VOLTAGE),
                    rel_tol=1e-9, abs_tol=1e-6,
                )
            ):
                raise RuntimeError(
                    'Nonzero target verification failed: '
                    f"requested {TEST_VOLTAGE:g} V, got {nonzero_state['target_v']} V"
                )
            nonzero_positive_adc = select_and_read_adc(
                device, MODULE_ADDRESS, False, current_high,
                safety_limit_v=NONZERO_ABS_LIMIT_V,
                abort_callback=abort_nonzero,
            )
            report['steps']['nonzero_positive_adc'] = nonzero_positive_adc
            guard_max_abs_voltage_v = max(
                guard_max_abs_voltage_v,
                validate_nonzero_adc('positive', nonzero_positive_adc),
            )
            nonzero_negative_adc = select_and_read_adc(
                device, MODULE_ADDRESS, True, current_high,
                safety_limit_v=NONZERO_ABS_LIMIT_V,
                abort_callback=abort_nonzero,
            )
            report['steps']['nonzero_negative_adc'] = nonzero_negative_adc
            guard_max_abs_voltage_v = max(
                guard_max_abs_voltage_v,
                validate_nonzero_adc('negative', nonzero_negative_adc),
            )
            report['steps']['nonzero_global_off_after_observation'] = record_call(
                device, 'nonzero_global_off_after_observation',
                device.set_enable, False, required=True,
            )
            report['steps']['nonzero_zero_target_after_observation'] = record_call(
                device, 'nonzero_zero_target_after_observation',
                device.set_hv_supply_target_output_voltage,
                MODULE_ADDRESS, 0.0, required=True,
            )
            report['steps']['nonzero_standby_after_observation'] = record_call(
                device, 'nonzero_standby_after_observation',
                device.set_module_activation_state,
                MODULE_ADDRESS, False, required=True,
            )
            time.sleep(SETTLE_SECONDS)
            nonzero_safe_state = read_hv_state(device, MODULE_ADDRESS)
            report['steps']['nonzero_safe_state_after_observation'] = (
                nonzero_safe_state
            )
            if (
                nonzero_safe_state['global_enabled'] is not False
                or nonzero_safe_state['target_v'] != 0.0
                or nonzero_safe_state['direct_activation'] is not False
                or nonzero_safe_state['pwm_activation'] is not False
            ):
                raise RuntimeError(
                    'Automatic shutdown after the guarded observation was not confirmed.'
                )
            report['cleanup']['discharge'] = wait_for_discharge()
            if not report['cleanup']['discharge']['confirmed']:
                raise RuntimeError(
                    f'Output did not discharge below {DISCHARGE_LIMIT_V:g} V '
                    f'within {DISCHARGE_TIMEOUT_SECONDS:g} seconds.'
                )
            print(
                'Output is OFF and discharged below 1 V. '
                'Enter the values recorded during the energized window.'
            )
            external_observation = {
                'positive_voltage_v': input(
                    'External meter on the positive connector, in V: '
                ).strip(),
                'negative_voltage_v': input(
                    'External meter on the negative connector, in V: '
                ).strip(),
                'module_led_color': input(
                    f'Physical module LED color during the {TEST_VOLTAGE:g} V window: '
                ).strip(),
            }
            report['external_observation'] = external_observation
            positive_adc_values = nonzero_positive_adc['state'][
                'measured_voltage'
            ].get('values', [])
            negative_adc_values = nonzero_negative_adc['state'][
                'measured_voltage'
            ].get('values', [])
            report['nonzero_summary'] = {
                'target_confirmed': True,
                'global_enabled': nonzero_state['global_enabled'],
                'direct_activation': nonzero_state['direct_activation'],
                'pwm_activation': nonzero_state['pwm_activation'],
                'control_active': nonzero_state['control_active'],
                'module_state_raw': nonzero_state['module_state_raw'],
                'pwm_voltage_set_v': nonzero_state['pwm_voltage_set_v'],
                'pwm_voltage_measured_v': nonzero_state['pwm_voltage_measured_v'],
                'module_led_rgb': nonzero_state['module_led_rgb'],
                'guard_sample_count': len(guard_samples),
                'guard_max_abs_voltage_v': guard_max_abs_voltage_v,
                'guard_abs_limit_v': float(NONZERO_ABS_LIMIT_V),
                'hold': report['nonzero_hold_summary'],
                'positive_adc_v': (
                    positive_adc_values[1]
                    if len(positive_adc_values) >= 2 else None
                ),
                'negative_adc_v': (
                    negative_adc_values[1]
                    if len(negative_adc_values) >= 2 else None
                ),
                'automatic_shutdown_confirmed': True,
                'external_observation': external_observation,
            }

        return report
    finally:
        if opened:
            if safety_sequence_started:
                report['cleanup']['global_off'] = record_call(
                    device, 'cleanup_global_off', device.set_enable, False
                )
                for address in (1, 2):
                    report['cleanup'][f'zero_target_{address}'] = record_call(
                        device, f'cleanup_zero_target_{address}',
                        device.set_hv_supply_target_output_voltage, address, 0.0
                    )
                    standby = record_call(
                        device, f'cleanup_standby_{address}',
                        device.set_module_activation_state, address, False
                    )
                    report['cleanup'][f'standby_{address}'] = standby
                    if standby.get('status') != device.NO_ERR:
                        report['cleanup'][f'standby_retry_{address}'] = record_call(
                            device, f'cleanup_standby_retry_{address}',
                            device.set_module_activation_state, address, False
                        )
                if initial_measurement_ranges is not None:
                    report['cleanup']['restore_measurement_ranges'] = record_call(
                        device, 'restore_measurement_ranges',
                        device.set_hv_supply_meas_ranges, MODULE_ADDRESS,
                        *initial_measurement_ranges,
                    )
                if (
                    temporary_config_attempted
                    and initial_config_safe
                    and initial_config_raw is not None
                ):
                    restore = record_call(
                        device, 'restore_initial_configuration',
                        device.set_current_config, initial_config_raw,
                    )
                    report['cleanup']['restore_initial_configuration'] = restore
                    if restore.get('status') != device.NO_ERR:
                        report['cleanup']['restore_initial_configuration_retry'] = (
                            record_call(
                                device, 'restore_initial_configuration_retry',
                                device.set_current_config, initial_config_raw,
                            )
                        )
                    try:
                        time.sleep(SETTLE_SECONDS)
                        restored = read_current_configuration(device)
                        restored['matches_initial_bytes'] = (
                            restored['raw_bytes'] == initial_config_raw
                        )
                        restored['safety_violations'] = safe_config_violations(
                            restored['decoded']
                        )
                        report['cleanup']['restored_configuration_readback'] = (
                            restored
                        )
                    except Exception as exc:
                        report['cleanup']['restored_configuration_readback'] = {
                            'error': f'{type(exc).__name__}: {exc}'
                        }
                if 'discharge' not in report['cleanup']:
                    report['cleanup']['discharge'] = wait_for_discharge()
                try:
                    report['cleanup']['final_state'] = read_hv_state(
                        device, MODULE_ADDRESS
                    )
                except Exception as exc:
                    report['cleanup']['final_state'] = {
                        'error': f'{type(exc).__name__}: {exc}'
                    }
            report['cleanup']['close'] = record_call(
                device, 'close_port', device.close_port
            )
        report_path = PLUGIN_DIR / f"esi_hv_activation_probe_{datetime.now():%Y%m%d_%H%M%S}.json"
        report_path.write_text(
            json.dumps(json_safe(report), indent=2), encoding='utf-8'
        )
        print(f'Report written to {report_path}')

activation_report = run_activation_probe()


## Interpretation

- `active_global_off_final.pwm_activation == true`: the global gate is not a prerequisite for changing the HVC toggle.
- `active_global_off_final.pwm_activation == false` followed by `baseline_global_on.pwm_activation == true`: the OFF-gate request was latched but the PWM readback hides it until `SetEnable(True)`.
- `active_global_off_final.pwm_activation == false`, `baseline_global_on.pwm_activation == false`, then `active_zero_v_final.pwm_activation == true`: `SetEnable(True)` must precede module activation; the plugin sequence must be changed.
- Any module activation request returning a nonzero status means the matching DLL/firmware path is not working. Do not arm the nonzero test.
- `temporary_configuration_readback.verification_violations` must be empty, with global OFF, every module gate OFF, every target at `0 V`, and only the selected module's `max_voltage_step_v` changed to `10.008`.
- `cleanup.restored_configuration_readback.matches_initial_bytes` must be true and `safety_violations` must be empty before authorizing another run.
- `active_zero_v_final.direct_activation` or `active_zero_v_final.pwm_activation` remains false after the global-ON request: the HVC toggle was not confirmed. Inspect `complete_state`, `interlock_state`, `interlock_enable`, `inputs`, `module_state_active`, and `module_led_rgb`.
- Yellow at `0 V` is expected here: both the notebook report and `ESI-Controller.exe` return RGB `[true, true, false]` while the direct and PWM activation bits can still be true. It does not invalidate the gate test.
- Every `nonzero_guard_samples` entry must remain at or below `nonzero_abs_limit_v`; a `nonzero_guard_abort_reason` means the automatic cutoff fired and no higher test is permitted.
- `nonzero_hold_summary.completed == true` confirms that the PWM measurement reached the `95 V` trigger and remained under continuous guards for the requested 20-second observation. The trigger is not an accuracy verdict; evaluate the recorded PWM, both ADCs, and both external meters.
- `cleanup.discharge.confirmed` must be true before the controller port is considered safely releasable after a nonzero attempt.
- If `initial_current_configuration.decoded.hv.<address>.max_voltage_step_v == 0` and remains zero in `nonzero_current_configuration`, compare it with the supplied active configurations (`10.008 V`). This would explain why direct target and gate writes succeed while regulation stays idle.
- At the guarded target, an inactive control bit or zero PWM set value means the target register accepted the request but regulation did not start. Compare both ADCs and external meters before changing plugin logic.
- Only one selected ADC state remains near zero: the earlier plugin display was reading the other polarity. `nonzero_positive_adc` and `nonzero_negative_adc` identify which one.
- A red physical module LED at a nonzero target is not by itself a fault code. CGC also uses red/blue to identify positive/negative outputs; interpret the LED together with the selected ADC polarity, `control_active`, PWM values, voltage/device state, and the external meter.
